# 05. CLI 提交完整链路

Notebook 里通过 `subprocess.run` 执行 CLI：写出 OriginIR 文件，`uniqc submit --platform dummy --wait`，
并展示返回结果。`--platform dummy` 默认对应无约束、无噪声的 `dummy`；可通过 `--backend virtual-line-3` 指定虚拟拓扑，或通过 `--backend originq:WK_C180` 走真实 backend compile/transpile + 本地含噪执行。真实发布检查时也可以把平台切换成云后端。


In [1]:
import pathlib

# Find PROJECT_ROOT by traversing up until we find .venv
_notebook_cwd = pathlib.Path.cwd()
for parent in [_notebook_cwd] + list(_notebook_cwd.parents):
    if (parent / ".venv").is_dir():
        PROJECT_ROOT = parent
        break
else:
    PROJECT_ROOT = _notebook_cwd

In [2]:
import subprocess
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="uniqc-bp-cli-"))
circuit_file = workdir / "bell.originir"
circuit_file.write_text(
    "QINIT 2\nCREG 2\nH q[0]\nCNOT q[0], q[1]\nMEASURE q[0], c[0]\nMEASURE q[1], c[1]\n",
    encoding="utf-8",
)

uniqc_bin = PROJECT_ROOT / ".venv" / "bin" / "uniqc"
uniqc_display = "./.venv/bin/uniqc"
cmd = f"{uniqc_bin} submit {circuit_file} -p dummy -s 64 --wait --format json"
completed = subprocess.run(cmd, shell=True, text=True, capture_output=True, check=True)
print("command:", uniqc_display, "submit", circuit_file, "-p dummy -s 64 --wait --format json")
print(completed.stdout)

command: ./.venv/bin/uniqc submit /tmp/uniqc-bp-cli-xypk4gur/bell.originir -p dummy -s 64 --wait --format json
{
  "task_id": "9dc41044252bdac8",
  "platform": "dummy",
  "shots": 64
}
{
  "00": 32,
  "11": 32
}

